Installations

In [1]:
!pip install -q faiss-cpu PyPDF2 sentence-transformers transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 22.4 MB/s eta 0:00:00


Imports and Setup

//

In [2]:
import os
import torch
import faiss
import PyPDF2
import numpy as np
from sentence_transformers import SentenceTransformer, util
from transformers import T5ForConditionalGeneration, T5Tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=device)


from transformers import T5ForConditionalGeneration, T5Tokenizer

t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-base").to(device)
t5_tokenizer = T5Tokenizer.from_pretrained("google/flan-t5-base")
t5_model.eval()

t5_model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


T5ForConditionalGeneration(
  (shared): Embedding(32128, 768)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 768)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=768, out_features=768, bias=False)
              (k): Linear(in_features=768, out_features=768, bias=False)
              (v): Linear(in_features=768, out_features=768, bias=False)
              (o): Linear(in_features=768, out_features=768, bias=False)
              (relative_attention_bias): Embedding(32, 12)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=768, out_features=2048, bias=False)
              (wi_1): Linear(in_features=768, out_features=2048, bias=False)
              (wo):

In [3]:
def extract_text_from_pdfs(pdf_paths):
    docs = []
    for path in pdf_paths:
        with open(path, 'rb') as f:
            reader = PyPDF2.PdfReader(f)
            for i, page in enumerate(reader.pages):
                text = page.extract_text()
                if text:
                    docs.append({
                        'pdf_name': os.path.basename(path),
                        'page_number': i + 1,
                        'text': text.strip()
                    })
    return docs


In [4]:
def build_faiss_index(docs):
    texts = [doc['text'] for doc in docs]
    embeddings = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=True)

    index = faiss.IndexFlatIP(embeddings.shape[1])
    faiss.normalize_L2(embeddings)
    index.add(embeddings)
    return index, embeddings, texts

In [5]:
def semantic_search(query, docs, embeddings, index, top_k=5):
    query_emb = embedder.encode(query, convert_to_tensor=True)
    query_emb_np = query_emb.cpu().numpy().reshape(1, -1)
    faiss.normalize_L2(query_emb_np)

    scores, indices = index.search(query_emb_np, top_k)
    top_results = []

    for i, idx in enumerate(indices[0]):
        doc = docs[idx]
        # Move the embeddings tensor to the same device as query_emb
        cos_sim = util.cos_sim(query_emb, torch.tensor(embeddings[idx], device=query_emb.device))[0][0].item()
        top_results.append({
            "pdf_name": doc["pdf_name"],
            "page_number": doc["page_number"],
            "text": doc["text"],
            "score": cos_sim
        })

    # Rerank by cosine similarity
    top_results = sorted(top_results, key=lambda x: x['score'], reverse=True)
    return top_results


In [13]:
def generate_answer(prompt, max_new_tokens=200):
    inputs = t5_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)
    input_ids = inputs.input_ids.to(device)
    attention_mask = inputs.attention_mask.to(device)
    with torch.no_grad():
        output_ids = t5_model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.7
        )
    return t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)


In [15]:
from google.colab import files
uploaded = files.upload()
pdf_paths = list(uploaded.keys())

# Extract, index, and search
docs = extract_text_from_pdfs(pdf_paths)
index, embeddings, texts = build_faiss_index(docs)

# User query
query = input("🔍 Enter your query: ")

# Semantic search
results = semantic_search(query, docs, embeddings, index)

# Prepare context for Flan-T5
context = " ".join([res["text"] for res in results[:3]])
prompt = f"""You are a helpful assistant. Based on the context provided, give a detailed and comprehensive answer to the question. Use complete sentences and explain thoroughly.

Context: {context}

Question: {query}

Detailed Answer:"""

# Generate answer
generated_answer = generate_answer(prompt, max_new_tokens=300)

# Display results
print("\n🔎 Query:", query)
print("="*60)
print("💡 Answer:", generated_answer)
print("\n📄 Top 3 Retrieved Results:\n")
for i, res in enumerate(results[:3], 1):
    print(f"{i}. PDF: {res['pdf_name']} | Page: {res['page_number']} | Score: {res['score']:.4f}")
    print(f"Excerpt: {res['text'][:300].strip()}...\n")